# Fooocus_extend – VSW Google Colab

Diese Fassung ist für Schulungen und Workshops vorbereitet und zeigt beim Start **Fortschritt, aktuelle Phase, Laufzeit und einen Zeit-Richtwert** an.

## Start
1. Öffne dieses Notebook direkt aus dem VSW-GitHub-Repository.
2. Klicke auf **▶ Fooocus_extend STARTEN / NEUSTARTEN**.
3. Beim allerersten Start können Paketversionen angepasst werden; Colab startet dann einmal neu.
4. Nach der Wiederverbindung dieselbe Start-Zelle erneut ausführen.
5. Sobald **BEREIT** und ein `gradio.live`-Link erscheint, diesen öffnen.

> **Wichtig:** Fooocus/Fooocus_extend ist mit der derzeitigen Abhängigkeitsbasis für **Python 3.11** vorgesehen. Dieses Notebook ist deshalb auf **Colab Runtime 2025.07 (Python 3.11.13)** festgelegt. Eine Python-3.12-Sitzung wird sofort abgebrochen, bevor lange Installationen beginnen.

## Datenhaltung
- Notebook: dauerhaft auf GitHub bzw. optional als Kopie in Google Drive.
- Programmumgebung: temporär unter `/content` in der Colab-VM.
- Bilder: standardmäßig temporär. Nur bei `GoogleDrive_output = True` werden Ergebnisse dauerhaft nach `MyDrive/outputs` geschrieben.

**Sicherheit:** Keine Passwörter, Tokens, API-Keys oder vertraulichen Inhalte in öffentliche Notebook-Ausgaben schreiben. Nach der Übung: **Laufzeit → Laufzeit trennen und löschen**.


In [ ]:
# @title ▶ Fooocus_extend STARTEN / NEUSTARTEN

import os
import sys
import time
import shutil
import queue
import threading
import subprocess
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

# ============================================================
# EINSTELLUNGEN
# ============================================================

Fooocus_Profile = "realistic" #@param ["default", "realistic", "anime"]
Fooocus_Theme = "dark" #@param ["dark", "light"]
Tunnel = "gradio" #@param ["gradio", "cloudflared"]
Memory_patch = True #@param {type:"boolean"}
GoogleDrive_output = False #@param {type:"boolean"}
Getestete_Version_verwenden = True #@param {type:"boolean"}

# Für den Schulungseinsatz getesteter Fooocus_extend-Stand (v9.3.5 / 09.09.2026)
PINNED_COMMIT = "7d32c923c172644023f77243bd7af4183ecb3737"
REPO_URL = "https://github.com/shaitanzx/Fooocus_extend.git"
REPO_DIR = Path("/content/Fooocus_extend")
OUTPUT_DIR = Path("/content/drive/MyDrive/outputs")
LOG_FILE = Path("/content/fooocus_extend_startup.log")
PORT = "7865"

START = time.monotonic()

def fmt_time(seconds):
    seconds = int(max(0, seconds))
    m, s = divmod(seconds, 60)
    h, m = divmod(m, 60)
    if h:
        return f"{h:d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

def status(percent, phase, detail="", eta=""):
    elapsed = fmt_time(time.monotonic() - START)
    line = f"[{percent:3d}%] {phase} | vergangen {elapsed}"
    if eta:
        line += f" | Restzeit-Richtwert {eta}"
    print("\n" + line)
    if detail:
        print("      " + detail)

def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None

def run_checked(cmd, desc):
    print(f"→ {desc}")
    result = subprocess.run(cmd, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"{desc} fehlgeschlagen (Exit-Code {result.returncode}).")

# ============================================================
# 1. RUNTIME / PYTHON PRÜFEN
# ============================================================

status(2, "Runtime prüfen", f"Python {sys.version.split()[0]}")

# Fooocus/Fooocus_extend setzt mit der aktuellen Abhängigkeitsbasis Python 3.11 voraus.
if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        f"Falsche Python-Version: {sys.version.split()[0]}\n\n"
        "Für diese VSW-Fassung wird Colab Runtime 2025.07 mit Python 3.11 benötigt.\n"
        "Python 3.12/3.13 führt bei Fooocus derzeit zu Abhängigkeitsfehlern und langen, "
        "letztlich erfolglosen Installationsversuchen.\n\n"
        "Bitte in Colab wählen:\n"
        "Laufzeit → Laufzeittyp ändern → Runtime-Version 2025.07 → GPU/T4\n"
        "Danach die Laufzeit neu verbinden und diese Zelle erneut starten."
    )

# ============================================================
# 2. GPU / SPEICHER PRÜFEN
# ============================================================

status(5, "GPU und Speicher prüfen")

gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True
)
if gpu_check.returncode != 0:
    raise RuntimeError(
        "Keine NVIDIA-GPU erkannt.\n"
        "Bitte Laufzeit → Laufzeittyp ändern → GPU auswählen."
    )
print("✓ GPU:", gpu_check.stdout.strip())

free_gb = shutil.disk_usage("/content").free / (1024**3)
print(f"✓ Freier Speicher: {free_gb:.1f} GB")
if free_gb < 12:
    raise RuntimeError(
        f"Zu wenig freier Speicher ({free_gb:.1f} GB). "
        "Für Installation, Modelle und temporäre Dateien sollten mindestens ca. 12 GB frei sein."
    )

# ============================================================
# 3. BASIS-ABHÄNGIGKEITEN
# ============================================================

status(10, "Basis-Abhängigkeiten prüfen", eta="ca. 1–3 Min. bei Änderungen")

required_packages = {
    "nvidia-cudnn-cu12": "9.1.0.70",
    "pygit2": "1.15.1",
    "numpy": "1.26.4",
}

to_install = []
for package, wanted in required_packages.items():
    current = installed_version(package)
    print(f"  {package}: {current or 'nicht installiert'}", end="")
    if current != wanted:
        print(f" → {wanted}")
        to_install.append(f"{package}=={wanted}")
    else:
        print(" ✓")

if to_install:
    print("\nEinmalige Vorbereitung läuft. Danach startet Colab automatisch neu.")
    cmd = [sys.executable, "-m", "pip", "install", "--progress-bar", "on", *to_install]
    result = subprocess.run(cmd, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            "Die Basis-Abhängigkeiten konnten nicht installiert werden.\n"
            "Prüfe zuerst, ob wirklich Runtime 2025.07 / Python 3.11 verwendet wird."
        )

    print("\n✓ Vorbereitung abgeschlossen.")
    print("Colab startet jetzt einmal neu. Nach der Wiederverbindung dieselbe START-Zelle erneut ausführen.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)

else:
    status(18, "Basis-Abhängigkeiten bereit")

    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "Die GPU ist vorhanden, aber PyTorch erkennt CUDA nicht.\n"
            "Bitte die Laufzeit einmal neu starten und diese Zelle erneut ausführen."
        )
    print("✓ CUDA:", torch.cuda.get_device_name(0))
    print("✓ PyTorch:", torch.__version__)

    # ========================================================
    # 4. ALTE PROZESSE BEENDEN
    # ========================================================

    status(20, "Alte Prozesse beenden")
    for pattern in ("entry_with_update.py", "launch.py", "cloudflared tunnel"):
        subprocess.run(
            ["pkill", "-f", pattern],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

    # ========================================================
    # 5. FOOOCUS_EXTEND BEREITSTELLEN
    # ========================================================

    status(24, "Fooocus_extend bereitstellen", eta="ca. 0–2 Min.")

    if not REPO_DIR.exists():
        clone_cmd = ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)]
        run_checked(clone_cmd, "Repository klonen")
    elif not (REPO_DIR / ".git").exists():
        raise RuntimeError("/content/Fooocus_extend existiert, ist aber kein gültiges Git-Repository.")

    if Getestete_Version_verwenden:
        current = subprocess.run(
            ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
            capture_output=True, text=True
        ).stdout.strip()
        if current != PINNED_COMMIT:
            run_checked(
                ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", PINNED_COMMIT],
                "Getesteten Programmstand laden"
            )
            run_checked(
                ["git", "-C", str(REPO_DIR), "checkout", "--force", PINNED_COMMIT],
                "Getesteten Programmstand aktivieren"
            )
        print("✓ Getesteter Fooocus_extend-Stand aktiv:", PINNED_COMMIT[:10])
    else:
        run_checked(
            ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", "main"],
            "Aktuellen main-Stand abrufen"
        )
        run_checked(
            ["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"],
            "Aktuellen main-Stand aktivieren"
        )
        print("✓ Aktueller main-Stand aktiv.")

    # ========================================================
    # 6. GOOGLE DRIVE OPTIONAL
    # ========================================================

    output_args = []
    if GoogleDrive_output:
        status(30, "Google Drive verbinden")
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive", force_remount=False)
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        output_args = ["--output-path", str(OUTPUT_DIR)]
        print("✓ Ausgabeordner:", OUTPUT_DIR)
    else:
        print("✓ Google Drive Output: AUS (temporäre Colab-Speicherung)")

    # ========================================================
    # 7. CLOUDFLARED OPTIONAL
    # ========================================================

    if Tunnel == "cloudflared":
        status(32, "Cloudflared vorbereiten", eta="ca. 1 Min.")
        if shutil.which("cloudflared") is None:
            deb_file = "/tmp/cloudflared-linux-amd64.deb"
            run_checked(
                ["wget", "--show-progress", "-O", deb_file,
                 "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"],
                "Cloudflared herunterladen"
            )
            run_checked(["dpkg", "-i", deb_file], "Cloudflared installieren")
        run_checked(
            [sys.executable, str(REPO_DIR / "patcher_tunel.py")],
            "Cloudflared-Patch anwenden"
        )

    # ========================================================
    # 8. FOOOCUS STARTEN – MIT LIVE-LOG, HEARTBEAT UND DIAGNOSE
    # ========================================================

    args = [sys.executable, "-u", "launch.py", "--port", PORT]

    if Fooocus_Profile == "realistic":
        args += ["--preset", "realistic"]
    elif Fooocus_Profile == "anime":
        args += ["--preset", "anime"]

    if Fooocus_Theme == "dark":
        args += ["--theme", "dark"]

    if Tunnel == "gradio":
        args += ["--share"]

    if Memory_patch:
        args += ["--always-high-vram", "--all-in-fp16"]

    args += output_args
    os.chdir(REPO_DIR)

    print("\n" + "=" * 72)
    print("FOOOCUS_EXTEND – STARTPROTOKOLL")
    print("=" * 72)
    print("Profil:              ", Fooocus_Profile)
    print("Theme:               ", Fooocus_Theme)
    print("Tunnel:              ", Tunnel)
    print("Memory Patch:        ", Memory_patch)
    print("Google Drive Output: ", GoogleDrive_output)
    print("Getestete Version:   ", Getestete_Version_verwenden)
    print("GPU:                 ", torch.cuda.get_device_name(0))
    print("Logdatei:            ", LOG_FILE)
    print("=" * 72)

    status(
        35,
        "Fooocus-Prozess startet",
        "Beim ersten Start werden Abhängigkeiten und mehrere Modelldateien geladen.",
        "typisch ca. 6–20 Min. beim ersten Start; Folgestarts deutlich schneller"
    )

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    proc = subprocess.Popen(
        args,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )

    q = queue.Queue()
    captured = []
    current_phase = {"name": "Start / Abhängigkeiten", "eta": "ca. 6–20 Min."}

    def reader():
        try:
            for line in iter(proc.stdout.readline, ""):
                q.put(line)
        finally:
            q.put(None)

    threading.Thread(target=reader, daemon=True).start()

    last_heartbeat = time.monotonic()
    stream_finished = False
    ready_seen = False

    def update_phase_from_line(line):
        low = line.lower()
        if "installing requirements" in low or ("installing " in low and "requirement" in low):
            current_phase["name"] = "Python-Pakete installieren"
            current_phase["eta"] = "meist ca. 2–8 Min."
        elif "downloading" in low or ("download" in low and (".safetensors" in low or ".pth" in low or ".bin" in low)):
            current_phase["name"] = "Modelle / Hilfsdateien herunterladen"
            current_phase["eta"] = "meist ca. 3–15 Min."
        elif "loading" in low and ("model" in low or "vae" in low):
            current_phase["name"] = "Modelle initialisieren"
            current_phase["eta"] = "meist ca. 1–4 Min."
        elif "running on local url" in low or "running on public url" in low or "gradio.live" in low:
            current_phase["name"] = "Weboberfläche bereit"
            current_phase["eta"] = "0 Min."

    with LOG_FILE.open("w", encoding="utf-8") as log:
        while True:
            try:
                item = q.get(timeout=1)
                if item is None:
                    stream_finished = True
                else:
                    captured.append(item)
                    log.write(item)
                    log.flush()
                    print(item, end="")
                    update_phase_from_line(item)
                    if "gradio.live" in item.lower() or "running on public url" in item.lower():
                        ready_seen = True
            except queue.Empty:
                pass

            now = time.monotonic()
            if now - last_heartbeat >= 20 and proc.poll() is None:
                elapsed = fmt_time(now - START)
                print(
                    f"\n… läuft weiter | Phase: {current_phase['name']} | "
                    f"vergangen {elapsed} | Restzeit-Richtwert {current_phase['eta']}"
                )
                last_heartbeat = now

            if stream_finished and proc.poll() is not None:
                break

    returncode = proc.wait()

    if returncode != 0:
        print("\n" + "!" * 72)
        print("START FEHLGESCHLAGEN – DIAGNOSE")
        print("!" * 72)
        print(f"Exit-Code: {returncode}")
        print(f"Python: {sys.version.split()[0]}")
        print(f"PyTorch: {torch.__version__}")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Vollständiges Log: {LOG_FILE}")
        print("\nLetzte Logzeilen:\n")
        tail = captured[-80:]
        print("".join(tail))

        joined = "".join(captured).lower()
        if "torch==2.1.0" in joined and "no matching distribution" in joined:
            print(
                "\nERKANNTER FEHLER: Torch 2.1.0 kann in dieser Python-Version nicht installiert werden.\n"
                "Bitte Runtime 2025.07 / Python 3.11 verwenden."
            )
        elif "numpy.core.multiarray failed to import" in joined or ("cupy" in joined and "importerror" in joined):
            print(
                "\nERKANNTER FEHLER: NumPy/CuPy-Kompatibilitätsproblem. "
                "Dies tritt besonders auf neueren Python-3.12-Colab-Runtimes auf.\n"
                "Bitte Runtime 2025.07 / Python 3.11 verwenden und die Sitzung neu starten."
            )
        elif "no space left on device" in joined:
            print("\nERKANNTER FEHLER: Zu wenig Speicherplatz in der Colab-VM.")

        raise RuntimeError(
            "Fooocus_extend wurde nicht erfolgreich gestartet. "
            "Bitte den Diagnoseblock oben bzw. /content/fooocus_extend_startup.log prüfen."
        )

    if ready_seen:
        status(100, "BEREIT", "Fooocus_extend wurde erfolgreich gestartet.", "0 Min.")
    else:
        print(
            "\nHinweis: Der Prozess wurde ohne Fehler beendet, aber es wurde kein "
            "öffentlicher Gradio-Link im Log erkannt."
        )


## Was bedeuten die Statusanzeigen?

- **0–20 %**: Runtime, GPU, CUDA und Abhängigkeiten werden geprüft.
- **20–35 %**: Fooocus_extend wird geklont bzw. auf den getesteten Stand gebracht.
- **35–55 %**: Python-Pakete werden geprüft bzw. installiert.
- **55–90 %**: Modelle und Hilfsdateien werden geladen bzw. initialisiert.
- **90–100 %**: Weboberfläche und Tunnel starten.

Die angezeigte Restzeit ist ein **Richtwert**, keine exakte Prognose: Downloadgeschwindigkeit, Colab-Auslastung und Modellcache schwanken stark.

Bei einem Fehler wird automatisch ein Diagnoseblock mit den letzten Logzeilen ausgegeben. Das vollständige Startprotokoll liegt zusätzlich unter `/content/fooocus_extend_startup.log`.
